In [1]:
from pathlib import Path
import h5py
import numpy as np
from structure_tensor.xdmf_io import write_xdmf_for_h5

In [2]:
_tomo = Path("../data/BIIAX_196(245)_02")

# { output_key: (source_file, source_key, target_dtype, pool_method) }
# pool_method: "avg" = block-average, "mode" = majority vote, "vec_avg" = sign-corrected normalized avg
file_map = {
    "volume": (_tomo / "BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vol.h5",      "volume", np.uint16,  "avg"),
    "seg":    (_tomo / "BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.seg.h5",      "seg_",   np.uint8,   "mode"),
    "vec":    (_tomo / "BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vec.half.h5", "vec",    np.float32, None),
}

crop      = None                      # None = full volume, e.g. np.s_[:100, :256, :256]
ds_factor = 4                         # 1 = no downsampling

output_path = _tomo / f"BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vol.seg.vec.size{ds_factor}.h5"

In [3]:
def _read_crop(ds, crop):
    """Read dataset with optional spatial crop — normalizes slices so h5py never sees None."""
    if crop is None:
        return ds[()]
    norm = tuple(slice(s.start or 0, s.stop, s.step or 1) for s in crop)
    return ds[norm] if ds.ndim == 3 else ds[(slice(None, None, 1),) + norm]


def _downsample(data, factor, method):
    if factor <= 1:
        return data
    Z, Y, X = data.shape[:3]
    Z2, Y2, X2 = Z // factor, Y // factor, X // factor
    d = data[:Z2*factor, :Y2*factor, :X2*factor]

    if method == "avg":
        # mean(axis=(1,3,5)) works directly on interleaved axes — no transpose needed
        if d.ndim == 3:
            return d.reshape(Z2, factor, Y2, factor, X2, factor).mean(axis=(1, 3, 5))
        return d.reshape(Z2, factor, Y2, factor, X2, factor, d.shape[-1]).mean(axis=(1, 3, 5))

    elif method == "mode":
        # must transpose [Z2,fz,Y2,fy,X2,fx] → [Z2,Y2,X2,fz,fy,fx] before flattening
        # otherwise the reshape scrambles which voxels belong to each block
        flat = (d.reshape(Z2, factor, Y2, factor, X2, factor)
                 .transpose(0, 2, 4, 1, 3, 5)
                 .reshape(Z2, Y2, X2, factor**3))
        n_labels = int(d.max()) + 1
        counts = np.apply_along_axis(lambda b: np.bincount(b, minlength=n_labels), -1, flat)
        return counts.argmax(axis=-1).astype(d.dtype)

    elif method == "vec_avg":
        # same transpose fix for (Z,Y,X,3) vector blocks
        flat = (d.reshape(Z2, factor, Y2, factor, X2, factor, 3)
                 .transpose(0, 2, 4, 1, 3, 5, 6)
                 .reshape(Z2, Y2, X2, factor**3, 3))
        ref = flat[..., :1, :]
        signs = np.sign((flat * ref).sum(axis=-1, keepdims=True))
        signs[signs == 0] = 1
        mean_vec = (flat * signs).mean(axis=-2)
        norm = np.linalg.norm(mean_vec, axis=-1, keepdims=True)
        norm[norm == 0] = 1
        return (mean_vec / norm).astype(d.dtype)

    else:  # stride
        s = (slice(None, None, factor),) * 3
        return d[s] if d.ndim == 3 else d[s + (slice(None),)]

In [4]:
with h5py.File(output_path, "a") as f_out:
    for out_key, (src_path, src_key, dtype, pool) in file_map.items():
        if out_key in f_out:
            del f_out[out_key]
        with h5py.File(src_path, "r") as f_in:
            ds = f_in[src_key]
            data = _read_crop(ds, crop).astype(dtype)
            if data.ndim == 4 and data.shape[0] == 3:
                data = np.moveaxis(data, 0, -1)   # (3,Z,Y,X) → (Z,Y,X,3)
            data = _downsample(data, ds_factor, pool)
            f_out.create_dataset(out_key, data=data)
            print(f"{src_path.name}:/{src_key} ({ds.dtype})\n\t"
                  f"→ {output_path.name}:/{out_key} ({np.dtype(dtype)})  {f_out[out_key].shape}"
                  )

keys = list(file_map.keys())
write_xdmf_for_h5(output_path, keys=keys, grid_key="volume")

BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vol.h5:/volume (uint16)
	→ BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vol.seg.vec.size4.h5:/volume (uint16)  (426, 345, 345)
BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.seg.h5:/seg_ (uint8)
	→ BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vol.seg.vec.size4.h5:/seg (uint8)  (426, 345, 345)
BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vec.half.h5:/vec (float16)
	→ BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vol.seg.vec.size4.h5:/vec (float32)  (426, 345, 345, 3)


PosixPath('../data/BIIAX_196(245)_02/BIIAX_196(245)_02_tomo-A_recon_crop_ztrim150.vol.seg.vec.size4.xdmf')